In [ ]:
import pandas as pd

df = pd.read_csv("s2022.csv")
print(df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 's2022.csv'

In [ ]:
import pandas as pd

df = pd.read_csv("s2022.csv")
print(df.shape)
df.head()

(1230, 26)


,Date,Start (ET),Visitor/Neutral,PTS,Home/Neutral,PTS.1,Box Score,OT?,Attend.,LOG,...,Home L,Away W,Away L,Home Conf,Away Conf,Same Conf?,Games Apart,Season Progress,Tier A,Third
0,"Tue, Oct 19, 2021",7:30p,Brooklyn Nets,104,Milwaukee Bucks,127,Box Score,NaN,"17,341",2:20,...,0,0,0,East,East,True,0.0,0.0,1,1st third
1,"Tue, Oct 19, 2021",10:00p,Golden State Warriors,121,Los Angeles Lakers,114,Box Score,NaN,"18,997",2:33,...,0,0,0,West,West,True,0.0,0.0,1,1st third
2,"Wed, Oct 20, 2021",7:00p,Indiana Pacers,122,Charlotte Hornets,123,Box Score,NaN,"15,521",2:20,...,0,0,0,East,East,True,0.0,0.0,1,1st third
3,"Wed, Oct 20, 2021",7:00p,Chicago Bulls,94,Detroit Pistons,88,Box Score,NaN,"20,088",2:09,...,0,0,0,East,East,True,0.0,0.0,1,1st third
4,"Wed, Oct 20, 2021",7:30p,Boston Celtics,134,New York Knicks,138,Box Score,2OT,"19,812",2:54,...,0,0,0,East,East,True,0.0,0.0,1,1st third


df["Date"] = pd.to_datetime(df["Date"])

df = df.rename(columns={
    "Visitor/Neutral": "away",
    "PTS": "away_pts",
    "Home/Neutral": "home",
    "PTS.1": "home_pts",
    "Tier A": "sheets_tier_a",
})

df = df[["Date", "away", "away_pts", "home", "home_pts", "sheets_tier_a"]]

print(df["Date"].min(), df["Date"].max())
print(df["sheets_tier_a"].sum())
df.head()

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])

df = df.rename(columns={
    "Visitor/Neutral": "away",
    "PTS": "away_pts",
    "Home/Neutral": "home",
    "PTS.1": "home_pts",
    "Tier A": "sheets_tier_a",
})

df = df[["Date", "away", "away_pts", "home", "home_pts", "sheets_tier_a"]]

print(df["Date"].min(), df["Date"].max())
print(df["sheets_tier_a"].sum())
df.head()

2021-10-19 00:00:00 2022-04-10 00:00:00
252


,Date,away,away_pts,home,home_pts,sheets_tier_a
0,2021-10-19,Brooklyn Nets,104,Milwaukee Bucks,127,1
1,2021-10-19,Golden State Warriors,121,Los Angeles Lakers,114,1
2,2021-10-20,Indiana Pacers,122,Charlotte Hornets,123,1
3,2021-10-20,Chicago Bulls,94,Detroit Pistons,88,1
4,2021-10-20,Boston Celtics,134,New York Knicks,138,1


In [ ]:
df = df.sort_values("Date").reset_index(drop=True)

teams = set(df["home"]) | set(df["away"])
wins = {t: 0 for t in teams}
losses = {t: 0 for t in teams}

home_w, home_l, away_w, away_l = [], [], [], []

for i in range(len(df)):
    h = df.loc[i, "home"]
    a = df.loc[i, "away"]

    home_w.append(wins[h])
    home_l.append(losses[h])
    away_w.append(wins[a])
    away_l.append(losses[a])

    if df.loc[i, "home_pts"] > df.loc[i, "away_pts"]:
        wins[h] += 1
        losses[a] += 1
    else:
        wins[a] += 1
        losses[h] += 1

df["home_w"] = home_w
df["home_l"] = home_l
df["away_w"] = away_w
df["away_l"] = away_l

print(wins["Cleveland Cavaliers"], losses["Cleveland Cavaliers"])
print(df.loc[1229, "home_w"] + df.loc[1229, "home_l"])
df.head()

44 38
81


,Date,away,away_pts,home,home_pts,sheets_tier_a,home_w,home_l,away_w,away_l
0,2021-10-19,Brooklyn Nets,104,Milwaukee Bucks,127,1,0,0,0,0
1,2021-10-19,Golden State Warriors,121,Los Angeles Lakers,114,1,0,0,0,0
2,2021-10-20,Denver Nuggets,110,Phoenix Suns,98,1,0,0,0,0
3,2021-10-20,Sacramento Kings,124,Portland Trail Blazers,121,1,0,0,0,0
4,2021-10-20,Orlando Magic,97,San Antonio Spurs,123,0,0,0,0,0


In [ ]:
east = {
    "Atlanta Hawks", "Boston Celtics", "Brooklyn Nets", "Charlotte Hornets",
    "Chicago Bulls", "Cleveland Cavaliers", "Detroit Pistons", "Indiana Pacers",
    "Miami Heat", "Milwaukee Bucks", "New York Knicks", "Orlando Magic",
    "Philadelphia 76ers", "Toronto Raptors", "Washington Wizards",
}
print(len(east & teams))

conf = {t: ("East" if t in east else "West") for t in teams}
df["home_conf"] = df["home"].map(conf)
df["away_conf"] = df["away"].map(conf)
df["same_conf"] = df["home_conf"] == df["away_conf"]

df["games_apart"] = ((df["home_w"] - df["home_l"]) - (df["away_w"] - df["away_l"])).abs() / 2

SEASON_LENGTH = 82
GAP = 2

df["tier_a"] = (df["same_conf"] & (df["games_apart"] <= GAP)).astype(int)
df["progress"] = (df["home_w"] + df["home_l"] + df["away_w"] + df["away_l"]) / 2 / SEASON_LENGTH
df["third"] = pd.cut(df["progress"], bins=[-0.01, 1/3, 2/3, 1.01],
                     labels=["1st", "2nd", "3rd"], right=False)

summary = df[df["same_conf"]].groupby("third", observed=True).agg(
    same_conf_games=("same_conf", "size"),
    tier_a_games=("tier_a", "sum"),
)
summary["tier_a_share"] = summary["tier_a_games"] / summary["same_conf_games"]
print(summary)

print((df["tier_a"] == df["sheets_tier_a"]).all())

15
       same_conf_games  tier_a_games  tier_a_share
third                                             
1st                266           158      0.593985
2nd                242            53      0.219008
3rd                272            41      0.150735
True


In [ ]:
def win_pct(t):
    games = wins[t] + losses[t]
    return wins[t] / games if games > 0 else 0

def conference_standings(conf_name):
    members = [t for t in teams if conf[t] == conf_name]
    return sorted(members, key=win_pct, reverse=True)

east_table = conference_standings("East")
for rank, t in enumerate(east_table, start=1):
    print(rank, t, wins[t], losses[t])


1 Miami Heat 53 29
2 Philadelphia 76ers 51 31
3 Milwaukee Bucks 51 31
4 Boston Celtics 51 31
5 Toronto Raptors 48 34
6 Chicago Bulls 46 36
7 Cleveland Cavaliers 44 38
8 Brooklyn Nets 44 38
9 Charlotte Hornets 43 39
10 Atlanta Hawks 43 39
11 New York Knicks 37 45
12 Washington Wizards 35 47
13 Indiana Pacers 25 57
14 Detroit Pistons 23 59
15 Orlando Magic 22 60


In [ ]:
def near(margin, line_margins):
    for m in line_margins:
        if abs(margin - m) / 2 <= GAP:
            return True
    return False

wins = {t: 0 for t in teams}
losses = {t: 0 for t in teams}

tier_b_list = []

for i in range(len(df)):
    h = df.loc[i, "home"]
    a = df.loc[i, "away"]

    if conf[h] == conf[a]:
        table = conference_standings(conf[h])
        margins = [wins[t] - losses[t] for t in table]
        line_6_7 = [margins[5], margins[6]]
        line_10_11 = [margins[9], margins[10]]

        h_m = wins[h] - losses[h]
        a_m = wins[a] - losses[a]

        both_6_7 = near(h_m, line_6_7) and near(a_m, line_6_7)
        both_10_11 = near(h_m, line_10_11) and near(a_m, line_10_11)

        if both_6_7 or both_10_11:
            tier_b_list.append(1)
        else:
            tier_b_list.append(0)
    else:
        tier_b_list.append(0)

    if df.loc[i, "home_pts"] > df.loc[i, "away_pts"]:
        wins[h] += 1
        losses[a] += 1
    else:
        wins[a] += 1
        losses[h] += 1

df["tier_b"] = tier_b_list

summary_b = df[df["same_conf"]].groupby("third", observed=True).agg(
    same_conf_games=("same_conf", "size"),
    tier_a_games=("tier_a", "sum"),
    tier_b_games=("tier_b", "sum"),
)
summary_b["tier_a_share"] = summary_b["tier_a_games"] / summary_b["same_conf_games"]
summary_b["tier_b_share"] = summary_b["tier_b_games"] / summary_b["same_conf_games"]
print(summary_b)
print((df.loc[~df["same_conf"], "tier_b"] == 0).all())

       same_conf_games  tier_a_games  tier_b_games  tier_a_share  tier_b_share
third                                                                         
1st                266           158           183      0.593985      0.687970
2nd                242            53            60      0.219008      0.247934
3rd                272            41            40      0.150735      0.147059
True


In [ ]:
def in_motion(team, gap, max_rank):
    table = conference_standings(conf[team])
    rank = table.index(team)
    if rank >= max_rank:
        return False
    margin = wins[team] - losses[team]
    neighbors = []
    if rank > 0:
        neighbors.append(table[rank - 1])
    if rank < len(table) - 1:
        neighbors.append(table[rank + 1])
    for n in neighbors:
        if abs(margin - (wins[n] - losses[n])) / 2 <= gap:
            return True
    return False

def seed_flags(gap, max_rank):
    for t in teams:
        wins[t] = 0
        losses[t] = 0
    either, both = [], []
    for i in range(len(df)):
        h = df.loc[i, "home"]
        a = df.loc[i, "away"]
        fh = in_motion(h, gap, max_rank)
        fa = in_motion(a, gap, max_rank)
        either.append(1 if (fh or fa) else 0)
        both.append(1 if (fh and fa) else 0)
        if df.loc[i, "home_pts"] > df.loc[i, "away_pts"]:
            wins[h] += 1
            losses[a] += 1
        else:
            wins[a] += 1
            losses[h] += 1
    return either, both

for gap in [1, 2]:
    either, both = seed_flags(gap, 11)
    df[f"seed_either_{gap}"] = either
    df[f"seed_both_{gap}"] = both

cols = ["seed_either_1", "seed_either_2", "seed_both_1", "seed_both_2"]
print(df.groupby("third", observed=True)[cols].mean().round(3))
print((df["seed_both_2"] <= df["seed_either_2"]).all())
print((df["seed_either_1"] <= df["seed_either_2"]).all())

       seed_either_1  seed_either_2  seed_both_1  seed_both_2
third                                                        
1st            0.940          0.950        0.486        0.519
2nd            0.847          0.922        0.360        0.450
3rd            0.697          0.859        0.228        0.407
True
True


In [ ]:
import pandas as pd

EAST = {
    "Atlanta Hawks", "Boston Celtics", "Brooklyn Nets", "Charlotte Hornets",
    "Chicago Bulls", "Cleveland Cavaliers", "Detroit Pistons", "Indiana Pacers",
    "Miami Heat", "Milwaukee Bucks", "New York Knicks", "Orlando Magic",
    "Philadelphia 76ers", "Toronto Raptors", "Washington Wizards",
}

def analyze_season(filename, season_length=82, gaps=(1, 2), max_rank=11):
    df = pd.read_csv(filename)
    df = df.rename(columns={
        "Visitor/Neutral": "away", "PTS": "away_pts",
        "Home/Neutral": "home", "PTS.1": "home_pts",
    })
    df = df[["Date", "away", "away_pts", "home", "home_pts"]]
    before = len(df)
    df = df.dropna(subset=["home", "away"])
    if len(df) != before:
        print(f"{filename}: dropped {before - len(df)} row(s) with no team names")
    if len(df) != 1230:
        raise ValueError(f"{filename}: expected 1230 games, found {len(df)}")
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)

    teams = set(df["home"]) | set(df["away"])
    if len(teams & EAST) != 15 or len(teams) != 30:
        raise ValueError(f"{filename}: team names don't match (East found: {len(teams & EAST)}, teams: {len(teams)})")
    conf = {t: ("East" if t in EAST else "West") for t in teams}
    wins = {t: 0 for t in teams}
    losses = {t: 0 for t in teams}

    def win_pct(t):
        games = wins[t] + losses[t]
        return wins[t] / games if games > 0 else 0

    def standings(conf_name):
        members = [t for t in teams if conf[t] == conf_name]
        return sorted(members, key=win_pct, reverse=True)

    def in_motion(team, gap):
        table = standings(conf[team])
        rank = table.index(team)
        if rank >= max_rank:
            return False
        margin = wins[team] - losses[team]
        neighbors = []
        if rank > 0:
            neighbors.append(table[rank - 1])
        if rank < len(table) - 1:
            neighbors.append(table[rank + 1])
        for n in neighbors:
            if abs(margin - (wins[n] - losses[n])) / 2 <= gap:
                return True
        return False

    rows = []
    for i in range(len(df)):
        h = df.loc[i, "home"]
        a = df.loc[i, "away"]
        row = {"progress": (wins[h] + losses[h] + wins[a] + losses[a]) / 2 / season_length}
        for gap in gaps:
            fh = in_motion(h, gap)
            fa = in_motion(a, gap)
            row[f"either_{gap}"] = int(fh or fa)
            row[f"both_{gap}"] = int(fh and fa)
        rows.append(row)

        if df.loc[i, "home_pts"] > df.loc[i, "away_pts"]:
            wins[h] += 1
            losses[a] += 1
        else:
            wins[a] += 1
            losses[h] += 1

    flags = pd.DataFrame(rows)
    flags["third"] = pd.cut(flags["progress"], bins=[-0.01, 1/3, 2/3, 1.01],
                            labels=["1st", "2nd", "3rd"], right=False)
    cols = [f"either_{g}" for g in gaps] + [f"both_{g}" for g in gaps]
    return flags.groupby("third", observed=True)[cols].mean()

results = {}
for name in ["s2022", "s2023", "s2024", "s2025", "s2026"]:
    results[name] = analyze_season(name + ".csv")
all_results = pd.concat(results, names=["season", "third"])
print(all_results.round(3))

s2023.csv: dropped 1 row(s) with no team names
              either_1  either_2  both_1  both_2
season third                                    
s2022  1st       0.940     0.950   0.486   0.519
       2nd       0.847     0.922   0.360   0.450
       3rd       0.697     0.859   0.228   0.407
s2023  1st       0.918     0.942   0.481   0.517
       2nd       0.877     0.908   0.420   0.493
       3rd       0.715     0.858   0.252   0.418
s2024  1st       0.918     0.932   0.482   0.511
       2nd       0.846     0.918   0.373   0.481
       3rd       0.771     0.868   0.267   0.392
s2025  1st       0.899     0.916   0.420   0.468
       2nd       0.786     0.881   0.272   0.396
       3rd       0.718     0.818   0.237   0.327
s2026  1st       0.897     0.921   0.404   0.457
       2nd       0.810     0.895   0.295   0.415
       3rd       0.714     0.853   0.177   0.316


In [ ]:
d = pd.read_csv("s2023.csv")
names = pd.concat([d["Visitor/Neutral"], d["Home/Neutral"]])
print(d.shape)
print(names.value_counts(dropna=False).tail(5))

(1231, 26)
Brooklyn Nets            82
Golden State Warriors    82
Atlanta Hawks            82
Miami Heat               82
NaN                       2
Name: count, dtype: int64


In [ ]:
print(d[d["Home/Neutral"].isna()])

     Date Start (ET) Visitor/Neutral  PTS Home/Neutral  PTS.1 Unnamed: 6  \
1230  NaN        NaN             NaN  NaN          NaN    NaN        NaN   

     Unnamed: 7 Attend.  LOG  ... Unnamed: 16  Unnamed: 17  Unnamed: 18  \
1230        NaN     NaN  NaN  ...           0            0            0   

     Unnamed: 19 Unnamed: 20 Unnamed: 21 Unnamed: 22 Unnamed: 23 Unnamed: 24  \
1230         NaN         NaN         NaN           0           0         NaN   

     Unnamed: 25  
1230   1st third  

[1 rows x 26 columns]


In [ ]:
print(m.sum().round(3))

NameError: name 'm' is not defined

In [ ]:
import pandas as pd

def team_margins(filename):
    df = pd.read_csv(filename)
    df = df.rename(columns={
        "Visitor/Neutral": "away", "PTS": "away_pts",
        "Home/Neutral": "home", "PTS.1": "home_pts",
    })
    df = df.dropna(subset=["home", "away"])
    df["home_margin"] = df["home_pts"] - df["away_pts"]

    h = df[["home", "home_margin"]].rename(columns={"home": "team", "home_margin": "margin"})
    a = df[["away"]].copy()
    a["margin"] = -df["home_margin"]
    a = a.rename(columns={"away": "team"})

    both = pd.concat([h, a])
    return both.groupby("team")["margin"].mean().sort_values(ascending=False)

m = team_margins("s2026.csv")
print(m.round(1))
print("sum:", m.sum().round(3))

team
Oklahoma City Thunder     11.1
San Antonio Spurs          8.3
Detroit Pistons            8.2
Boston Celtics             7.7
New York Knicks            6.3
Houston Rockets            5.2
Denver Nuggets             5.1
Charlotte Hornets          4.8
Cleveland Cavaliers        4.1
Minnesota Timberwolves     3.4
Toronto Raptors            2.8
Atlanta Hawks              2.4
Miami Heat                 2.3
Los Angeles Lakers         1.8
Phoenix Suns               1.5
Los Angeles Clippers       1.1
Orlando Magic              0.6
Philadelphia 76ers        -0.2
Portland Trail Blazers    -0.3
Golden State Warriors     -0.6
New Orleans Pelicans      -4.5
Chicago Bulls             -5.2
Dallas Mavericks          -5.5
Memphis Grizzlies         -6.0
Milwaukee Bucks           -6.2
Indiana Pacers            -8.0
Utah Jazz                 -8.4
Brooklyn Nets            -10.0
Sacramento Kings         -10.0
Washington Wizards       -12.0
Name: margin, dtype: float64
sum: -0.0


In [ ]:
def game_stats(filename):
    df = pd.read_csv(filename)
    df = df.rename(columns={
        "Visitor/Neutral": "away", "PTS": "away_pts",
        "Home/Neutral": "home", "PTS.1": "home_pts",
    })
    df = df.dropna(subset=["home", "away"])
    margin = df["home_pts"] - df["away_pts"]
    return margin.mean(), margin.std()

for name in ["s2022", "s2023", "s2024", "s2025", "s2026"]:
    avg, sd = game_stats(name + ".csv")
    print(name, "home edge:", round(avg, 2), " spread:", round(sd, 2))

s2022 home edge: 1.72  spread: 15.31
s2023 home edge: 2.5  spread: 13.67
s2024 home edge: 2.15  spread: 15.64
s2025 home edge: 1.63  spread: 15.89
s2026 home edge: 1.74  spread: 16.41


In [ ]:
def game_stats(filename):
    df = pd.read_csv(filename)
    df = df.rename(columns={
        "Visitor/Neutral": "away", "PTS": "away_pts",
        "Home/Neutral": "home", "PTS.1": "home_pts",
    })
    df = df.dropna(subset=["home", "away"])
    margin = df["home_pts"] - df["away_pts"]
    return margin.mean(), margin.std()

for name in ["s2022", "s2023", "s2024", "s2025", "s2026"]:
    avg, sd = game_stats(name + ".csv")
    print(name, "home edge:", round(avg, 2), " spread:", round(sd, 2))

s2022 home edge: 1.72  spread: 15.31
s2023 home edge: 2.5  spread: 13.67
s2024 home edge: 2.15  spread: 15.64
s2025 home edge: 1.63  spread: 15.89
s2026 home edge: 1.74  spread: 16.41


In [ ]:
def leftover_spread(filename):
    m = team_margins(filename)
    df = pd.read_csv(filename)
    df = df.rename(columns={
        "Visitor/Neutral": "away", "PTS": "away_pts",
        "Home/Neutral": "home", "PTS.1": "home_pts",
    })
    df = df.dropna(subset=["home", "away"])
    expected = df["home"].map(m) - df["away"].map(m)
    actual = df["home_pts"] - df["away_pts"]
    left = actual - expected
    return left.mean(), left.std()

for name in ["s2022", "s2023", "s2024", "s2025", "s2026"]:
    avg, sd = leftover_spread(name + ".csv")
    print(name, "home edge:", round(avg, 2), " leftover spread:", round(sd, 2))

s2022 home edge: 1.72  leftover spread: 13.8
s2023 home edge: 2.5  leftover spread: 12.53
s2024 home edge: 2.13  leftover spread: 13.68
s2025 home edge: 1.6  leftover spread: 13.63
s2026 home edge: 1.74  leftover spread: 14.05


In [ ]:
import numpy as np

def read_games(filename):
    df = pd.read_csv(filename)
    df = df.rename(columns={
        "Visitor/Neutral": "away", "PTS": "away_pts",
        "Home/Neutral": "home", "PTS.1": "home_pts",
    })
    return df.dropna(subset=["home", "away"])

def count_wins(df, home_win, teams):
    hw = df.loc[home_win, "home"].value_counts()
    aw = df.loc[~home_win, "away"].value_counts()
    return hw.add(aw, fill_value=0).reindex(teams, fill_value=0).astype(int)

def fake_season_wins(filename, edge=2.0, spread=13.5, seed=1):
    rng = np.random.default_rng(seed)
    m = team_margins(filename)
    df = read_games(filename)
    expected = df["home"].map(m) - df["away"].map(m) + edge
    fake_margin = expected + rng.normal(0, spread, len(df))
    return count_wins(df, fake_margin > 0, m.index)

filename = "s2026.csv"
df = read_games(filename)
teams = team_margins(filename).index
real = count_wins(df, df["home_pts"] > df["away_pts"], teams)
fake = fake_season_wins(filename)

table = pd.DataFrame({"real": real, "fake": fake}).sort_values("real", ascending=False)
print(table.head(5))
print(table.tail(5))
print("spread of real win totals:", round(real.std(), 2))
print("spread of fake win totals:", round(fake.std(), 2))

                       real  fake
team                             
Oklahoma City Thunder    64    67
San Antonio Spurs        62    54
Detroit Pistons          60    65
Boston Celtics           56    51
Denver Nuggets           54    50
                    real  fake
team                          
Utah Jazz             22    22
Sacramento Kings      22    14
Brooklyn Nets         20    20
Indiana Pacers        19    22
Washington Wizards    17    12
spread of real win totals: 13.84
spread of fake win totals: 14.01


In [ ]:
spreads = [fake_season_wins("s2026.csv", seed=s).std() for s in range(500)]
print("average spread:", round(np.mean(spreads), 2))
print("lowest / highest:", round(min(spreads), 2), "/", round(max(spreads), 2))

average spread: 13.95
lowest / highest: 12.45 / 15.84


In [ ]:
def analyze_games(df, season_length=82, gaps=(1, 2), max_rank=11):
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date", kind="stable").reset_index(drop=True)
    homes = df["home"].tolist()
    aways = df["away"].tolist()
    home_wins = df["home_win"].tolist()

    teams = set(homes) | set(aways)
    conf = {t: ("East" if t in EAST else "West") for t in teams}
    wins = {t: 0 for t in teams}
    losses = {t: 0 for t in teams}

    def win_pct(t):
        games = wins[t] + losses[t]
        return wins[t] / games if games > 0 else 0

    def standings(conf_name):
        members = [t for t in teams if conf[t] == conf_name]
        return sorted(members, key=win_pct, reverse=True)

    def in_motion(team, gap):
        table = standings(conf[team])
        rank = table.index(team)
        if rank >= max_rank:
            return False
        margin = wins[team] - losses[team]
        neighbors = []
        if rank > 0:
            neighbors.append(table[rank - 1])
        if rank < len(table) - 1:
            neighbors.append(table[rank + 1])
        for n in neighbors:
            if abs(margin - (wins[n] - losses[n])) / 2 <= gap:
                return True
        return False

    rows = []
    for h, a, hw in zip(homes, aways, home_wins):
        row = {"progress": (wins[h] + losses[h] + wins[a] + losses[a]) / 2 / season_length}
        for gap in gaps:
            fh = in_motion(h, gap)
            fa = in_motion(a, gap)
            row[f"either_{gap}"] = int(fh or fa)
            row[f"both_{gap}"] = int(fh and fa)
        rows.append(row)
        if hw:
            wins[h] += 1
            losses[a] += 1
        else:
            wins[a] += 1
            losses[h] += 1

    flags = pd.DataFrame(rows)
    flags["third"] = pd.cut(flags["progress"], bins=[-0.01, 1/3, 2/3, 1.01],
                            labels=["1st", "2nd", "3rd"], right=False)
    cols = [f"either_{g}" for g in gaps] + [f"both_{g}" for g in gaps]
    return flags.groupby("third", observed=True)[cols].mean()

def real_games(filename):
    df = read_games(filename).copy()
    df["home_win"] = df["home_pts"] > df["away_pts"]
    return df[["Date", "away", "home", "home_win"]]

def fake_games(filename, edge=2.0, spread=13.5, seed=0):
    rng = np.random.default_rng(seed)
    m = team_margins(filename)
    df = read_games(filename)[["Date", "away", "home"]].copy()
    expected = df["home"].map(m) - df["away"].map(m) + edge
    df["home_win"] = (expected + rng.normal(0, spread, len(df))) > 0
    return df

def last_third_both1(df):
    return analyze_games(df).loc["3rd", "both_1"]

files = ["s2022.csv", "s2023.csv", "s2024.csv", "s2025.csv", "s2026.csv"]

real_vals = [last_third_both1(real_games(f)) for f in files]
print("real, 5-season average:", round(np.mean(real_vals), 3))

fake_vals = []
for f in files:
    for s in range(20):
        fake_vals.append(last_third_both1(fake_games(f, seed=s)))
print("fake, 100 seasons average:", round(np.mean(fake_vals), 3))
print("fake lowest / highest:", round(min(fake_vals), 3), "/", round(max(fake_vals), 3))

real, 5-season average: 0.236
fake, 100 seasons average: 0.222
fake lowest / highest: 0.077 / 0.415


In [ ]:
def trim_to_72(df, seed=0, remove_each=10):
    rng = np.random.default_rng(seed)
    homes = df["home"].tolist()
    aways = df["away"].tolist()
    teams = sorted(set(homes) | set(aways))
    for attempt in range(200):
        removed = {t: 0 for t in teams}
        drop = []
        for i in rng.permutation(len(df)):
            h, a = homes[i], aways[i]
            if removed[h] < remove_each and removed[a] < remove_each:
                removed[h] += 1
                removed[a] += 1
                drop.append(i)
        if all(v == remove_each for v in removed.values()):
            return df.drop(df.index[drop])
    raise ValueError("could not build a 72-game schedule")

sim82, sim72_half, sim72_one = [], [], []
for f in files:
    for s in range(30):
        g = fake_games(f, seed=s)
        sim82.append(analyze_games(g, season_length=82, gaps=(1,)).loc["3rd", "both_1"])
        t = trim_to_72(g, seed=s)
        r72 = analyze_games(t, season_length=72, gaps=(0.5, 1.0)).loc["3rd"]
        sim72_half.append(r72["both_0.5"])
        sim72_one.append(r72["both_1.0"])

print("sim 82 games, gap 1:   ", round(np.mean(sim82), 3))
print("sim 72 games, gap 0.5: ", round(np.mean(sim72_half), 3))
print("sim 72 games, gap 1.0: ", round(np.mean(sim72_one), 3))

sim 82 games, gap 1:    0.226
sim 72 games, gap 0.5:  0.14
sim 72 games, gap 1.0:  0.253


In [ ]:
def trim_by_rule(df, rule, seed=0, remove_each=10):
    rng = np.random.default_rng(seed)
    homes = df["home"].tolist()
    aways = df["away"].tolist()
    same_conf = [(h in EAST) == (a in EAST) for h, a in zip(homes, aways)]
    teams = sorted(set(homes) | set(aways))
    if rule == "cross_first":
        eligible = [not s for s in same_conf]
    elif rule == "conf_only":
        eligible = same_conf
    else:
        eligible = [True] * len(df)
    for attempt in range(500):
        removed = {t: 0 for t in teams}
        drop = []
        for i in rng.permutation(len(df)):
            if not eligible[i]:
                continue
            h, a = homes[i], aways[i]
            if removed[h] < remove_each and removed[a] < remove_each:
                removed[h] += 1
                removed[a] += 1
                drop.append(i)
        if all(v == remove_each for v in removed.values()):
            return df.drop(df.index[drop])
    raise ValueError("could not build a 72-game schedule")

results = {"cross_first": [], "conf_only": []}
for rule in results:
    for f in files:
        for s in range(20):
            g = fake_games(f, seed=s)
            t = trim_by_rule(g, rule, seed=s)
            r = analyze_games(t, season_length=72, gaps=(0.5, 1.0)).loc["3rd"]
            results[rule].append((r["both_0.5"], r["both_1.0"]))
    arr = np.array(results[rule])
    print(rule, "gap 0.5:", round(arr[:, 0].mean(), 3), " gap 1.0:", round(arr[:, 1].mean(), 3))

cross_first gap 0.5: 0.142  gap 1.0: 0.252
conf_only gap 0.5: 0.141  gap 1.0: 0.253


In [ ]:
def analyze_games2(df, season_length=82, gaps=(1,), max_rank=11):
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date", kind="stable").reset_index(drop=True)
    homes = df["home"].tolist()
    aways = df["away"].tolist()
    home_wins = df["home_win"].tolist()

    teams = set(homes) | set(aways)
    conf = {t: ("East" if t in EAST else "West") for t in teams}
    wins = {t: 0 for t in teams}
    losses = {t: 0 for t in teams}

    def win_pct(t):
        games = wins[t] + losses[t]
        return wins[t] / games if games > 0 else 0

    def standings(conf_name):
        members = [t for t in teams if conf[t] == conf_name]
        return sorted(members, key=win_pct, reverse=True)

    def in_motion(team, gap):
        table = standings(conf[team])
        rank = table.index(team)
        if rank >= max_rank:
            return False
        margin = wins[team] - losses[team]
        neighbors = []
        if rank > 0:
            neighbors.append(table[rank - 1])
        if rank < len(table) - 1:
            neighbors.append(table[rank + 1])
        for n in neighbors:
            if abs(margin - (wins[n] - losses[n])) / 2 <= gap:
                return True
        return False

    rows = []
    for h, a, hw in zip(homes, aways, home_wins):
        row = {"progress": (wins[h] + losses[h] + wins[a] + losses[a]) / 2 / season_length}
        same = conf[h] == conf[a]
        diff = abs((wins[h] - losses[h]) - (wins[a] - losses[a])) / 2
        for gap in gaps:
            row[f"both_{gap}"] = int(in_motion(h, gap) and in_motion(a, gap))
            row[f"direct_{gap}"] = int(same and diff <= gap)
        rows.append(row)
        if hw:
            wins[h] += 1
            losses[a] += 1
        else:
            wins[a] += 1
            losses[h] += 1

    flags = pd.DataFrame(rows)
    flags["third"] = pd.cut(flags["progress"], bins=[-0.01, 1/3, 2/3, 1.01],
                            labels=["1st", "2nd", "3rd"], right=False)
    cols = [f"both_{g}" for g in gaps] + [f"direct_{g}" for g in gaps]
    return flags.groupby("third", observed=True)[cols].mean()

def last_third(df, season_length, gaps):
    return analyze_games2(df, season_length, gaps).loc["3rd"]

real82 = [last_third(real_games(f), 82, (1,))["direct_1"] for f in files]
print("real 82, direct gap 1:", round(np.mean(real82), 3))

rules = ["random", "cross_first", "conf_only"]
out = {"fake82": []}
for r in rules:
    out[r] = []
for f in files:
    for s in range(20):
        g = fake_games(f, seed=s)
        out["fake82"].append(last_third(g, 82, (1,))["direct_1"])
        for r in rules:
            t = trim_by_rule(g, r, seed=s)
            x = last_third(t, 72, (0.5, 1.0))
            out[r].append((x["direct_0.5"], x["direct_1.0"]))

print("fake 82, direct gap 1:", round(np.mean(out["fake82"]), 3))
for r in rules:
    arr = np.array(out[r])
    print(r, "gap 0.5:", round(arr[:, 0].mean(), 3), " gap 1.0:", round(arr[:, 1].mean(), 3))

real 82, direct gap 1: 0.051
fake 82, direct gap 1: 0.042
random gap 0.5: 0.028  gap 1.0: 0.05
cross_first gap 0.5: 0.031  gap 1.0: 0.055
conf_only gap 0.5: 0.025  gap 1.0: 0.043


In [ ]:
def end_block(df, n_last, seed=0):
    if n_last == 0:
        return df
    rng = np.random.default_rng(seed)
    homes = df["home"].tolist()
    aways = df["away"].tolist()
    same_conf = [(h in EAST) == (a in EAST) for h, a in zip(homes, aways)]
    teams = sorted(set(homes) | set(aways))
    for attempt in range(500):
        count = {t: 0 for t in teams}
        chosen = []
        for i in rng.permutation(len(df)):
            if not same_conf[i]:
                continue
            h, a = homes[i], aways[i]
            if count[h] < n_last and count[a] < n_last:
                count[h] += 1
                count[a] += 1
                chosen.append(i)
        if all(v == n_last for v in count.values()):
            out = df.copy()
            out["Date"] = pd.to_datetime(out["Date"])
            late = out["Date"].max() + pd.Timedelta(days=1)
            out.iloc[chosen, out.columns.get_loc("Date")] = late
            return out
    raise ValueError("could not build the end block")

for n_last in [0, 6, 10, 14]:
    direct_vals, both_vals = [], []
    for f in files:
        for s in range(20):
            g = fake_games(f, seed=s)
            b = end_block(g, n_last, seed=s)
            x = last_third(b, 82, (1,))
            direct_vals.append(x["direct_1"])
            both_vals.append(x["both_1"])
    print("last", n_last, "same-conf | direct-race:", round(np.mean(direct_vals), 3),
          " seed-in-motion:", round(np.mean(both_vals), 3))

last 0 same-conf | direct-race: 0.042  seed-in-motion: 0.222
last 6 same-conf | direct-race: 0.047  seed-in-motion: 0.222
last 10 same-conf | direct-race: 0.048  seed-in-motion: 0.225
last 14 same-conf | direct-race: 0.051  seed-in-motion: 0.222


In [ ]:
def lock_games(df, ks=(6, 10)):
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date", kind="stable").reset_index(drop=True)
    homes = df["home"].tolist()
    aways = df["away"].tolist()
    home_wins = df["home_win"].tolist()

    total = {}
    for t in homes + aways:
        total[t] = total.get(t, 0) + 1
    conf = {t: ("East" if t in EAST else "West") for t in total}
    members = {c: [t for t in total if conf[t] == c] for c in ("East", "West")}
    wins = {t: 0 for t in total}
    played = {t: 0 for t in total}
    left = {(c, k): None for c in members for k in ks}

    for h, a, hw in zip(homes, aways, home_wins):
        played[h] += 1
        played[a] += 1
        if hw:
            wins[h] += 1
        else:
            wins[a] += 1
        for c in {conf[h], conf[a]}:
            for k in ks:
                if left[(c, k)] is not None:
                    continue
                table = sorted(members[c], key=lambda t: wins[t], reverse=True)
                top = table[:k]
                rest = table[k:]
                best_outside = max(wins[t] + total[t] - played[t] for t in rest)
                if min(wins[t] for t in top) > best_outside:
                    left[(c, k)] = sum(total[t] - played[t] for t in members[c]) / len(members[c])

    out = {}
    for k in ks:
        vals = [left[(c, k)] if left[(c, k)] is not None else 0 for c in members]
        out[k] = sum(vals) / len(vals)
    return out

variants = {
    "82 games": lambda g, s: (g, 82),
    "82 + block 10": lambda g, s: (end_block(g, 10, seed=s), 82),
    "72 random": lambda g, s: (trim_by_rule(g, "random", seed=s), 72),
}
for name, make in variants.items():
    k6, k10 = [], []
    for f in files:
        for s in range(20):
            g = fake_games(f, seed=s)
            d, length = make(g, s)
            r = lock_games(d)
            k6.append(r[6] / length)
            k10.append(r[10] / length)
    print(name, "| share of season left when top-6 locks:", round(np.mean(k6), 3),
          " top-10 locks:", round(np.mean(k10), 3))

82 games | share of season left when top-6 locks: 0.016  top-10 locks: 0.032
82 + block 10 | share of season left when top-6 locks: 0.016  top-10 locks: 0.031
72 random | share of season left when top-6 locks: 0.016  top-10 locks: 0.031


In [ ]:
def match_by_standings(table, k, rng):
    n = len(table)
    pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
    for attempt in range(2000):
        noise = 0.3 + attempt * 0.01
        order = sorted(pairs, key=lambda p: (p[1] - p[0]) + rng.uniform(0, noise))
        deg = [0] * n
        chosen = []
        for i, j in order:
            if deg[i] < k and deg[j] < k:
                deg[i] += 1
                deg[j] += 1
                chosen.append((table[i], table[j], j - i))
        if all(d == k for d in deg):
            return chosen
    raise ValueError("could not match teams by standings")

def finish_season(filename, k, seed=0, edge=2.0, spread=13.5):
    rng = np.random.default_rng(seed + 1000)
    m = team_margins(filename)
    g = fake_games(filename, seed=seed)
    if k == 0:
        return g, 0.0
    base = trim_by_rule(g, "conf_only", seed=seed, remove_each=k).copy()
    base["Date"] = pd.to_datetime(base["Date"])
    wins = {t: 0 for t in m.index}
    for h, a, hw in zip(base["home"], base["away"], base["home_win"]):
        wins[h if hw else a] += 1

    games = []
    gaps = []
    east = [t for t in m.index if t in EAST]
    west = [t for t in m.index if t not in EAST]
    for members in (east, west):
        order = sorted(members, key=lambda t: (-wins[t], rng.random()))
        for t1, t2, gap in match_by_standings(order, k, rng):
            if rng.random() < 0.5:
                home, away = t1, t2
            else:
                home, away = t2, t1
            expected = m[home] - m[away] + edge
            games.append((home, away, expected + rng.normal(0, spread) > 0))
            gaps.append(gap)

    last = base["Date"].max()
    rows = []
    for idx, i in enumerate(rng.permutation(len(games))):
        home, away, hw = games[i]
        rows.append({"Date": last + pd.Timedelta(days=1 + idx // 14),
                     "away": away, "home": home, "home_win": hw})
    final = pd.concat([base[["Date", "away", "home", "home_win"]], pd.DataFrame(rows)],
                      ignore_index=True)
    return final, float(np.mean(gaps))

for k in [0, 2, 4, 6, 8, 10]:
    direct_vals, both_vals, gap_vals = [], [], []
    for f in files:
        for s in range(15):
            final, avg_gap = finish_season(f, k, seed=s)
            x = last_third(final, 82, (1,))
            direct_vals.append(x["direct_1"])
            both_vals.append(x["both_1"])
            gap_vals.append(avg_gap)
    print("K =", k, "| direct-race:", round(np.mean(direct_vals), 3),
          " seed-in-motion:", round(np.mean(both_vals), 3),
          " avg seed distance of matched games:", round(np.mean(gap_vals), 2))

K = 0 | direct-race: 0.043  seed-in-motion: 0.222  avg seed distance of matched games: 0.0
K = 2 | direct-race: 0.065  seed-in-motion: 0.239  avg seed distance of matched games: 1.87
K = 4 | direct-race: 0.068  seed-in-motion: 0.239  avg seed distance of matched games: 2.61
K = 6 | direct-race: 0.064  seed-in-motion: 0.241  avg seed distance of matched games: 3.24
K = 8 | direct-race: 0.064  seed-in-motion: 0.243  avg seed distance of matched games: 3.92
K = 10 | direct-race: 0.057  seed-in-motion: 0.235  avg seed distance of matched games: 4.57
